In [1]:
# Imports

import base64
import json
import re
from io import BytesIO

import numpy as np
import torch
import gradio as gr
from openai import OpenAI
from PIL import Image
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
from kokoro import KPipeline

/Users/samarthmn/Projects/samarthmn/zyra/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Initialisation

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if torch.cuda.is_available() else torch.float32

# GPT Model
OLLAMA_MODEL = "gemma4:e4b"
gpt_model = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

# ASR Model
ASR_MODEL = "openai/whisper-large-v3-turbo"

asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    ASR_MODEL, dtype=dtype, low_cpu_mem_usage=True, use_safetensors=True
)
asr_model.to(device)

processor = AutoProcessor.from_pretrained(ASR_MODEL)

pipe_asr = pipeline(
    "automatic-speech-recognition",
    model=asr_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    dtype=dtype,
    device=device,
)

# TTS Model
TTS_MODEL = "hexgrad/Kokoro-82M"
TTS_VOICE = "af_heart"
TTS_SAMPLE_RATE = 24000
tts_pipeline = KPipeline(lang_code="a", repo_id=TTS_MODEL, device=device)

Loading weights: 100%|██████████| 587/587 [00:00<00:00, 872.57it/s] 
/Users/samarthmn/Projects/samarthmn/zyra/.venv/lib/python3.14/site-packages/torch/nn/modules/rnn.py:1009: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)
/Users/samarthmn/Projects/samarthmn/zyra/.venv/lib/python3.14/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


In [3]:
system_prompt = """
You are Zyra, a friendly plant care assistant. Identify plants, diagnose common problems
(yellow leaves, pests, over/under watering), and give practical care advice including
watering and fertilizing frequency.

Rules:
- Only identify a plant when you are highly confident. If unsure, set common_name and
  scientific_name to null, confidence_percent below 50, and say "I don't know" in advice.
- If you need clearer photos (different angles, leaves, flowers, stems, or soil), set
  needs_more_info to true and explain exactly what to photograph in more_info_request.
- Keep advice to 1-2 short, clear sentences.

Respond with valid JSON only (no markdown fences), using this schema:
{
  "common_name": string or null,
  "scientific_name": string or null,
  "confidence_percent": integer from 0 to 100,
  "advice": string,
  "needs_more_info": boolean,
  "more_info_request": string or null
}
"""

In [ ]:
def image_to_base64(image: np.ndarray) -> str:
    pil_image = Image.fromarray(image.astype("uint8"), "RGB")
    buffer = BytesIO()
    pil_image.save(buffer, format="JPEG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")


def transcribe_audio(audio) -> str:
    if audio is None:
        return ""
    sample_rate, audio_array = audio
    result = pipe_asr({"array": audio_array, "sampling_rate": sample_rate})
    return result["text"].strip()


def text_to_speech(text: str):
    if not text or not text.strip():
        return None

    chunks = []
    for result in tts_pipeline(text, voice=TTS_VOICE):
        if result.audio is not None:
            chunks.append(result.audio.cpu().numpy())

    if not chunks:
        return None

    return (TTS_SAMPLE_RATE, np.concatenate(chunks))


def parse_plant_response(raw: str) -> dict:
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", raw, re.DOTALL)
        if match:
            try:
                return json.loads(match.group())
            except json.JSONDecodeError:
                pass

    return {
        "common_name": None,
        "scientific_name": None,
        "confidence_percent": 0,
        "advice": raw,
        "needs_more_info": False,
        "more_info_request": None,
    }


def format_plant_identification(data: dict) -> str:
    common = data.get("common_name") or "Unknown"
    scientific = data.get("scientific_name") or "Unknown"
    confidence = int(data.get("confidence_percent") or 0)
    return (
        f"Common name: {common}\n"
        f"Scientific name: {scientific}\n"
        f"Confidence: {confidence}%"
    )


def format_advice_text(data: dict) -> str:
    advice = (data.get("advice") or "").strip()
    if data.get("needs_more_info") and data.get("more_info_request"):
        advice = f"{advice}\n\n{data['more_info_request'].strip()}"
    return advice


def multi_model_input(
    image,
    text,
    audio,
    messages,
):
    if image is None and not (text and text.strip()) and audio is None:
        return (
            None,
            "Please provide a plant photo, a question, or a voice message.",
            "Common name: Unknown\nScientific name: Unknown\nConfidence: 0%",
            messages,
        )

    user_text = (text or "").strip()
    spoken_text = transcribe_audio(audio)
    if spoken_text:
        user_text = f"{user_text} {spoken_text}".strip() if user_text else spoken_text

    if not user_text and image is None:
        return (
            None,
            "I couldn't understand the audio. Please try again or type your question.",
            "Common name: Unknown\nScientific name: Unknown\nConfidence: 0%",
            messages,
        )

    if messages is None or len(messages) == 0:
        messages = [{"role": "system", "content": system_prompt}]

    if image is not None:
        image_b64 = image_to_base64(image)
        content = [
            {
                "type": "text",
                "text": user_text or "Identify this plant and give brief care advice.",
            },
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"},
            },
        ]
    else:
        content = user_text

    messages.append({"role": "user", "content": content})

    try:
        raw_response = (
            gpt_model.chat.completions.create(model=OLLAMA_MODEL, messages=messages)
            .choices[0]
            .message.content
        )
    except Exception as exc:
        error = f"Sorry, I couldn't reach the plant care model: {exc}"
        return (
            None,
            error,
            "Common name: Unknown\nScientific name: Unknown\nConfidence: 0%",
            messages,
        )

    plant_data = parse_plant_response(raw_response)
    advice_text = format_advice_text(plant_data)
    plant_info = format_plant_identification(plant_data)

    messages.append({"role": "assistant", "content": advice_text})
    audio_output = text_to_speech(advice_text)

    return audio_output, advice_text, plant_info, messages


with gr.Blocks(title="Zyra — Plant Care Assistant") as demo:
    gr.Markdown(
        "# Zyra\n"
        "Your multi-modal plant care assistant. Upload a photo, type a question, "
        "or record a voice message."
    )

    chat_history = gr.State([])

    with gr.Row():
        with gr.Column():
            image_input = gr.Image(label="Plant Photo", type="numpy")
            text_input = gr.Textbox(
                label="Question",
                placeholder="e.g. Why are the leaves turning yellow?",
            )
            audio_input = gr.Audio(label="Voice Input", type="numpy")
            submit_btn = gr.Button("Ask Zyra", variant="primary")
        with gr.Column():
            plant_info_output = gr.Textbox(label="Plant Identification", lines=3)
            text_output = gr.Textbox(label="Zyra's Advice", lines=4)
            audio_output = gr.Audio(label="Listen to Advice")

    submit_btn.click(
        fn=multi_model_input,
        inputs=[image_input, text_input, audio_input, chat_history],
        outputs=[audio_output, text_output, plant_info_output, chat_history],
    )

demo.launch(inline=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


/Users/samarthmn/Projects/samarthmn/zyra/.venv/lib/python3.14/site-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
/Users/samarthmn/Projects/samarthmn/zyra/.venv/lib/python3.14/site-packages/gradio/processing_utils.py:724: UserWarning: Trying to convert audio automatically from float32 to 16-bit int format.
  warnings.warn(warning.format(data.dtype))
